# Components walkthrough

This notebook executes the existing progressive examples in deterministic mock mode. Each step adds one component; the implementation remains in `agentic_tutorial.education`.

In [1]:
from agentic_tutorial.education import run_tutorial_async

component_names = (
    "basic-model",
    "tool-use",
    "explicit-state",
    "planning",
    "retained-context",
    "critique-validation",
    "bounded-tracing",
)
component_results = {name: await run_tutorial_async(name) for name in component_names}
[(name, result["concept"]) for name, result in component_results.items()]

[('basic-model', 'model invocation'),
 ('tool-use', 'tool use'),
 ('explicit-state', 'explicit state'),
 ('planning', 'planning'),
 ('retained-context', 'retained context'),
 ('critique-validation', 'critique and validation'),
 ('bounded-tracing', 'bounded execution and tracing')]

The sequence moves from one canonical model response to validated tool use, explicit state, a finite plan, retained message context, critique, and finally a bounded trace. The expected trace ends with `termination`.

In [2]:
assert component_results["basic-model"]["answer"] == "Paper paper-001 is relevant."
assert component_results["planning"]["bounded_steps"] == 3
assert component_results["critique-validation"]["valid_initially"] is False
assert component_results["bounded-tracing"]["event_types"][-1] == "termination"
component_results

{'basic-model': {'concept': 'model invocation',
  'answer': 'Paper paper-001 is relevant.'},
 'tool-use': {'concept': 'tool use',
  'result': [{'paper_id': 'paper-001',
    'title': 'Evaluating Agent Trajectories',
    'topic': 'agent evaluation'}]},
 'explicit-state': {'concept': 'explicit state',
  'run_id': 'state-example-run',
  'message_count': 1},
 'planning': {'concept': 'planning',
  'plan': ('search local catalogue',
   'inspect evidence',
   'write concise answer'),
  'bounded_steps': 3},
 'retained-context': {'concept': 'retained context',
  'answer': 'paper-001',
  'message_count': 3},
 'critique-validation': {'concept': 'critique and validation',
  'valid_initially': False,
  'revised': 'Paper paper-001 is relevant.'},
 'bounded-tracing': {'concept': 'bounded execution and tracing',
  'event_types': ['run_start', 'budget', 'termination'],
  'deterministic_trace_bytes': 773}}

## Human approval

The existing approval example checkpoints an exact proposed action before execution. This deterministic rejection path demonstrates that the consequential simulated tool is not run.

In [3]:
from agentic_tutorial.education.approval import run_approval_demo
from agentic_tutorial.schemas import TerminationStatus

approval_state, executed = await run_approval_demo("reject", "Unused revised title")
assert executed == []
assert approval_state.termination is not None
assert approval_state.termination.status is TerminationStatus.FAILURE

These examples are deliberately small. They teach component boundaries, not the full research-assistant workflow; generated traces and checkpoints are written beneath the ignored `outputs/runs/` directory.